# Passing callback to `synthetmic.DiagramGenerator.fit` method

In this example, we will show how to pass callback function to
`synthetmic.DiagramGenerator.fit` method, particularly,
`synthetmic.LaguerreDiagramGenerator.fit` (the same approach
can be used for `synthetmic.VoronoiDiagramGenerator.fit`).

Callback function can be used to track the progress of the algorithm
implemented in the above-mentioned method. In this example, we will use
the popular `tqdm` package to track the progress of our microstructure
generation.

We will create a 3D Laguerre diagram with all cells or grains having the same volumes.


We first install `synthetmic` and `tqdm`.

In [ ]:
!pip install synthetmic tqdm matplotlib

Next, we import the necessary classes and functtions.

In [5]:
from synthetmic import LaguerreDiagramGenerator, LaguerreEvent
from synthetmic.data.toy import create_data_with_constant_volumes
import matplotlib.pyplot as plt
from tqdm import tqdm

We now create configuration for the diagram. Below, we create a unit cube with 10,000 grains.
All grains will have the volume $1/10000$ cubic units.

> Note that you can always create your own custom configuration and not required to
> use the provided functions in `synthetmic`.

In [16]:
config = create_data_with_constant_volumes(
    space_dim=3,
    n_grains=10_000,
    is_periodic=False,
    random_state=42
)
config

DiagramConfig(domain=array([[0, 1],
       [0, 1],
       [0, 1]]), seeds=array([[0.37454012, 0.95071431, 0.73199394],
       [0.59865848, 0.15601864, 0.15599452],
       [0.05808361, 0.86617615, 0.60111501],
       ...,
       [0.77707099, 0.15825956, 0.12330421],
       [0.2693165 , 0.95227545, 0.74782421],
       [0.01945586, 0.40100484, 0.25739798]], shape=(10000, 3)), phases=array([0, 0, 0, ..., 0, 0, 0], shape=(10000,)), periodic=(False, False, False), volumes=array([0.0001, 0.0001, 0.0001, ..., 0.0001, 0.0001, 0.0001],
      shape=(10000,)), initial_weights=array([0., 0., 0., ..., 0., 0., 0.], shape=(10000,)))

We create a callback function to monitor the progress of the `fit` method.

In [ ]:
N_ITER = 30

pbar = tqdm(total=N_ITER, desc="Running")
def callback(e: LaguerreEvent) -> None:
        pbar.n = e.iteration
        pbar.set_postfix(
                {
                        "mean_percentage_error": e.mean_percentage_error,
                        "max_percentage_error": e.max_percentage_error,
                },
                refresh=True
        )
        pbar.refresh()

diagram = LaguerreDiagramGenerator(
    tol=1.0,
    n_iter=N_ITER,
    damp_param=1.0,
)
diagram.fit(config, callback=callback)
pbar.close()

Callback is not limited to showing iteration progress but also can be used to accumulate iteration metrics. The accumulated metrics can  then be processed further, for instance, plotting the metrics against iterations.

We domonstrate this below.

In [ ]:
history = {
    "Iteration": [],
    "Mean percentage error": [],
    "Max percentage error": []
}
def callback(e: LaguerreEvent) -> None:
    history["Iteration"].append(e.iteration)
    history["Mean percentage error"].append(e.mean_percentage_error)
    history["Max percentage error"].append(e.max_percentage_error)

diagram = LaguerreDiagramGenerator(
    tol=1.0,
    n_iter=N_ITER,
    damp_param=1.0,
)
diagram.fit(config, callback=callback)

In [ ]:
_, ax = plt.subplots()
for err in ("Mean percentage error", "Max percentage error"):
    ax.plot(
        history["Iteration"],
        history[err],
        linewidth=2,
        label=err,
        marker="o",
        markerfacecolor='white',
    )

ax.set_yscale('log')
ax.set_xlabel("Iteration")
ax.set_ylabel("Error (%)")
ax.legend()
